# Modelo de Previsão Mensal de Sinistro SUSEP

**Objetivo / Objective**

Construir, comparar e avaliar modelos para prever o valor de sinistro do próximo mês, usando a tabela wide de features:

`SUSEP_GOLD.TB_ML_SINISTRO_MENSAL_WIDE_FEATURES`

A granularidade do modelo é exclusivamente **ano/mês**, sem considerar empresa, ramo, grupo econômico ou UF.

---

## Modelos contemplados / Covered models

1. Baseline último mês  
2. Baseline sazonal do ano anterior  
3. Baseline média móvel 3 meses  
4. Baseline média móvel 6 meses  
5. Baseline média móvel 12 meses  
6. Baseline média ponderada 3 meses  
7. Linear Regression  
8. Ridge Regression  
9. Lasso Regression  
10. Random Forest  
11. XGBoost  
12. LightGBM  

---

## Estratégia / Strategy

- Usar backtesting temporal.
- Treinar sempre no passado.
- Prever sempre o próximo mês.
- Comparar todos os modelos pelas mesmas métricas.
- Escolher o modelo campeão.
- Treinar o modelo campeão com toda a base disponível.
- Gerar previsão para o próximo mês sem target observado.
- Registrar resultados em tabelas Delta.

# COMMAND ----------

## 1. Configuração inicial / Initial setup

Nesta etapa definimos parâmetros globais, tabelas de origem/destino e configurações gerais do experimento.

In this step we define global parameters, source/target tables and general experiment settings.

In [0]:
%pip install xgboost lightgbm

In [0]:
# Databricks notebook source
 ## 1. Configuração inicial / Initial setup

# COMMAND ----------



import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# Optional models
# Modelos opcionais: o notebook tenta importar, mas segue funcionando se a lib não estiver instalada.
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print(f"XGBoost não disponível / XGBoost not available: {e}")

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as e:
    LIGHTGBM_AVAILABLE = False
    print(f"LightGBM não disponível / LightGBM not available: {e}")

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except Exception as e:
    MLFLOW_AVAILABLE = False
    print(f"MLflow não disponível / MLflow not available: {e}")

# Source and target tables
# Tabelas de origem e destino
SOURCE_TABLE = "susep_gold.wt_ml_sinistro_mensal"

RESULTS_TABLE = "susep_gold.tb_ml_sinistro_mensal_model_results"
PREDICTIONS_TABLE = "susep_gold.tb_ml_sinistro_mensal_predictions"
FORECAST_TABLE = "susep_gold.tb_ml_sinistro_mensal_forecast_next_month"

# Target columns
# Colunas alvo
TARGET_COL = "vl_sinistro_target_prox_mes"
TARGET_LOG_COL = "ln_vl_sinistro_target_prox_mes"

# Main modeling flag
# Flag principal de modelagem
TRAIN_FLAG_COL = "fl_modelagem_12m"
FORECAST_FLAG_COL = "fl_linha_previsao"

# Backtesting configuration
# Configuração do backtesting
MIN_TRAINING_MONTHS = 48

# Prediction target mode
# 'raw'  -> modela diretamente o valor de sinistro
# 'log'  -> modela log1p(sinistro) e depois retorna expm1
TARGET_MODE = "raw"

# MLflow experiment
# Experimento MLflow
EXPERIMENT_NAME = "/Shared/susep_previsao_sinistro_mensal"

print("Configuração concluída / Setup completed")

# COMMAND ----------

## 2. Leitura da wide table / Read wide feature table

Lemos a tabela Gold gerada no passo anterior e convertemos para Pandas.

We read the Gold feature table created in the previous step and convert it to Pandas.

In [0]:
## 2. Leitura da wide table / Read wide feature table

# COMMAND ----------

df_spark = spark.table(SOURCE_TABLE)

display(df_spark.orderBy("dt_mes_referencia"))

df = df_spark.toPandas()

# Ensure date columns are datetime
# Garantir que as colunas de data estejam como datetime
for col in ["dt_mes_referencia", "dt_mes_previsao"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df = df.sort_values("dt_mes_referencia").reset_index(drop=True)

print(f"Quantidade de linhas / Number of rows: {len(df)}")
print(f"Período referência / Reference period: {df['nr_ano_mes_referencia'].min()} até {df['nr_ano_mes_referencia'].max()}")
print(f"Período previsão / Forecast period: {df['nr_ano_mes_previsao'].min()} até {df['nr_ano_mes_previsao'].max()}")

# COMMAND ----------

## 3. Validação básica dos dados / Basic data validation

Antes de modelar, validamos se a tabela possui linha de previsão, linhas de treino e target.

Before modeling, we validate whether the table has forecast row, training rows and target values.

In [0]:
 ## 3. Validação básica dos dados / Basic data validation

# COMMAND ----------

required_columns = [
    "nr_ano_mes_referencia",
    "dt_mes_referencia",
    "nr_ano_mes_previsao",
    "dt_mes_previsao",
    TARGET_COL,
    TRAIN_FLAG_COL,
    FORECAST_FLAG_COL
]

missing_required = [c for c in required_columns if c not in df.columns]

if missing_required:
    raise ValueError(f"Colunas obrigatórias ausentes / Missing required columns: {missing_required}")

# Training and forecast datasets
# Bases de treino e previsão
df_train_all = df[(df[TRAIN_FLAG_COL] == 1) & (df[TARGET_COL].notna())].copy()
df_forecast = df[df[FORECAST_FLAG_COL] == 1].copy()

if df_train_all.empty:
    raise ValueError("Não há linhas disponíveis para treino. Verifique fl_modelagem_12m e target.")

if df_forecast.empty:
    print("Aviso: não há linha marcada para previsão. O notebook ainda avaliará modelos, mas não gerará previsão final.")
else:
    print("Linha de previsão encontrada / Forecast row found:")
    display(spark.createDataFrame(df_forecast))

print(f"Linhas para treino / Training rows: {len(df_train_all)}")
print(f"Linhas para previsão / Forecast rows: {len(df_forecast)}")

# Check duplicates by reference month
# Checar duplicidade por mês referência
dup_count = df["nr_ano_mes_referencia"].duplicated().sum()
print(f"Duplicidades por mês referência / Duplicates by reference month: {dup_count}")

# COMMAND ----------

## 4. Definição das features / Feature selection

Aqui definimos grupos de variáveis para modelos lineares e modelos de árvore.

Here we define feature groups for linear models and tree-based models.

In [0]:
## 4. Definição das features / Feature selection

# COMMAND ----------

# Features mais enxutas e estáveis para regressões lineares
# Lean and stable features for linear regression models
linear_features = [
    "vl_sinistro_lag_0",
    "vl_sinistro_lag_1",
    "vl_sinistro_lag_2",
    "vl_sinistro_lag_3",
    "vl_sinistro_lag_6",
    "vl_sinistro_lag_11",
    "vl_sinistro_lag_12",
    "vl_sinistro_media_3m",
    "vl_sinistro_media_6m",
    "vl_sinistro_media_12m",
    "vl_premio_lag_0",
    "vl_premio_lag_1",
    "vl_premio_lag_3",
    "vl_premio_lag_12",
    "vl_premio_media_3m",
    "vl_premio_media_12m",
    "vl_sinistralidade_lag_0",
    "vl_sinistralidade_lag_1",
    "vl_sinistralidade_lag_12",
    "ft_mes_previsao_sin",
    "ft_mes_previsao_cos",
    "nr_indice_tempo_previsao"
]

# Features mais amplas para Random Forest, XGBoost e LightGBM
# Wider feature set for Random Forest, XGBoost and LightGBM
tree_features = [
    "nr_mes_previsao",
    "nr_trimestre_previsao",
    "nr_semestre_previsao",
    "nr_indice_tempo_previsao",
    "nr_indice_tempo_previsao_sq",

    "ft_mes_previsao_sin",
    "ft_mes_previsao_cos",
    "ft_mes_referencia_sin",
    "ft_mes_referencia_cos",

    "vl_sinistro_lag_0",
    "vl_sinistro_lag_1",
    "vl_sinistro_lag_2",
    "vl_sinistro_lag_3",
    "vl_sinistro_lag_4",
    "vl_sinistro_lag_5",
    "vl_sinistro_lag_6",
    "vl_sinistro_lag_11",
    "vl_sinistro_lag_12",
    "vl_sinistro_lag_13",
    "vl_sinistro_lag_24",

    "vl_premio_lag_0",
    "vl_premio_lag_1",
    "vl_premio_lag_2",
    "vl_premio_lag_3",
    "vl_premio_lag_4",
    "vl_premio_lag_5",
    "vl_premio_lag_6",
    "vl_premio_lag_11",
    "vl_premio_lag_12",
    "vl_premio_lag_13",
    "vl_premio_lag_24",

    "vl_sinistralidade_lag_0",
    "vl_sinistralidade_lag_1",
    "vl_sinistralidade_lag_2",
    "vl_sinistralidade_lag_3",
    "vl_sinistralidade_lag_6",
    "vl_sinistralidade_lag_11",
    "vl_sinistralidade_lag_12",
    "vl_sinistralidade_lag_24",

    "vl_sinistro_media_3m",
    "vl_sinistro_media_6m",
    "vl_sinistro_media_12m",
    "vl_sinistro_media_24m",
    "vl_sinistro_std_3m",
    "vl_sinistro_std_6m",
    "vl_sinistro_std_12m",
    "vl_sinistro_min_12m",
    "vl_sinistro_max_12m",

    "vl_premio_media_3m",
    "vl_premio_media_6m",
    "vl_premio_media_12m",
    "vl_premio_media_24m",
    "vl_premio_std_12m",
    "vl_premio_min_12m",
    "vl_premio_max_12m",

    "vl_sinistralidade_media_3m",
    "vl_sinistralidade_media_6m",
    "vl_sinistralidade_media_12m",
    "vl_sinistralidade_std_12m",

    "pc_sinistro_mom",
    "pc_sinistro_var_3m",
    "pc_sinistro_var_6m",
    "pc_sinistro_yoy_mes_referencia",
    "pc_sinistro_vs_mes_previsao_ano_anterior",

    "pc_premio_mom",
    "pc_premio_var_3m",
    "pc_premio_var_6m",
    "pc_premio_yoy_mes_referencia",

    "pc_sinistralidade_mom",
    "pc_sinistralidade_yoy_mes_referencia",

    "vl_sinistro_gap_media_3m",
    "vl_sinistro_gap_media_6m",
    "vl_sinistro_gap_media_12m",
    "pc_sinistro_gap_media_12m",
    "pc_premio_gap_media_12m",
    "pc_sinistralidade_gap_media_12m",

    "vl_sinistro_zscore_12m_excl_mes_atual",
    "fl_sinistro_outlier_12m",
    "fl_variacao_sinistro_mom_alta",
    "fl_variacao_sinistro_yoy_alta",

    "fl_mes_previsao_janeiro",
    "fl_mes_previsao_fevereiro",
    "fl_mes_previsao_marco",
    "fl_mes_previsao_abril",
    "fl_mes_previsao_maio",
    "fl_mes_previsao_junho",
    "fl_mes_previsao_julho",
    "fl_mes_previsao_agosto",
    "fl_mes_previsao_setembro",
    "fl_mes_previsao_outubro",
    "fl_mes_previsao_novembro",
    "fl_mes_previsao_dezembro"
]

# Keep only features available in the table
# Manter somente features existentes na tabela
linear_features = [c for c in linear_features if c in df.columns]
tree_features = [c for c in tree_features if c in df.columns]

print(f"Linear features: {len(linear_features)}")
print(f"Tree features: {len(tree_features)}")

print("\nLinear features:")
print(linear_features)

print("\nTree features:")
print(tree_features)

# COMMAND ----------

## 5. Funções de métricas / Metric functions

Definimos as métricas para avaliação dos modelos:

- MAE
- RMSE
- MAPE
- WAPE
- R²

We define model evaluation metrics.

In [0]:
 ## 5. Funções de métricas / Metric functions

# COMMAND ----------

def safe_mape(y_true, y_pred):
    """
    Calcula MAPE evitando divisão por zero.
    Calculates MAPE avoiding division by zero.
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def safe_wape(y_true, y_pred):
    """
    Calcula WAPE = soma dos erros absolutos / soma dos valores reais.
    Calculates WAPE = sum absolute errors / sum actuals.
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom

def calc_metrics(y_true, y_pred):
    """
    Retorna dicionário de métricas.
    Returns dictionary of metrics.
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = safe_mape(y_true, y_pred)
    wape = safe_wape(y_true, y_pred)

    # R² can be unstable with few observations, but useful as support
    # R² pode ser instável com poucas observações, mas é útil como apoio
    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = np.nan

    return {
        "vl_mae": float(mae),
        "vl_rmse": float(rmse),
        "vl_mape": float(mape),
        "vl_wape": float(wape),
        "vl_r2": float(r2) if not pd.isna(r2) else np.nan
    }

def inverse_target(y_pred):
    """
    Converte predição para valor original quando target log é usado.
    Converts prediction back to original value when log target is used.
    """
    y_pred = np.array(y_pred, dtype=float)
    if TARGET_MODE == "log":
        return np.expm1(y_pred)
    return y_pred

print("Funções de métricas criadas / Metric functions created")

# COMMAND ----------

## 6. Avaliação dos baselines / Baseline evaluation

Os baselines são importantes porque qualquer modelo mais sofisticado precisa ser melhor do que eles.

Baselines are important because any advanced model must beat them.

In [0]:
 ## 6. Avaliação dos baselines / Baseline evaluation

# COMMAND ----------

baseline_columns = {
    "baseline_ultimo_mes": "vl_pred_baseline_ultimo_mes",
    "baseline_sazonal_ano_anterior": "vl_pred_baseline_sazonal_ano_anterior",
    "baseline_media_3m": "vl_pred_baseline_media_3m",
    "baseline_media_6m": "vl_pred_baseline_media_6m",
    "baseline_media_12m": "vl_pred_baseline_media_12m",
    "baseline_media_ponderada_3m": "vl_pred_baseline_media_ponderada_3m"
}

baseline_results = []
baseline_predictions = []

for model_name, pred_col in baseline_columns.items():
    if pred_col not in df_train_all.columns:
        print(f"Baseline ignorado, coluna ausente / Skipped baseline, missing column: {pred_col}")
        continue

    tmp = df_train_all[[ 
        "nr_ano_mes_referencia",
        "dt_mes_referencia",
        "nr_ano_mes_previsao",
        "dt_mes_previsao",
        TARGET_COL,
        pred_col
    ]].dropna().copy()

    if tmp.empty:
        print(f"Baseline sem linhas válidas / Baseline with no valid rows: {model_name}")
        continue

    metrics = calc_metrics(tmp[TARGET_COL], tmp[pred_col])

    baseline_results.append({
        "nm_modelo": model_name,
        "ds_tipo_modelo": "baseline",
        "qt_observacoes_avaliacao": len(tmp),
        **metrics,
        "dh_execucao": datetime.now()
    })

    tmp_pred = tmp.rename(columns={pred_col: "vl_sinistro_previsto"})
    tmp_pred["nm_modelo"] = model_name
    tmp_pred["ds_tipo_modelo"] = "baseline"
    tmp_pred["vl_erro"] = tmp_pred[TARGET_COL] - tmp_pred["vl_sinistro_previsto"]
    tmp_pred["vl_erro_abs"] = np.abs(tmp_pred["vl_erro"])
    tmp_pred["pc_erro"] = np.where(
        tmp_pred[TARGET_COL] != 0,
        tmp_pred["vl_erro"] / tmp_pred[TARGET_COL],
        np.nan
    )
    tmp_pred["dh_execucao"] = datetime.now()

    baseline_predictions.append(tmp_pred)

df_baseline_results = pd.DataFrame(baseline_results)
df_baseline_predictions = pd.concat(baseline_predictions, ignore_index=True) if baseline_predictions else pd.DataFrame()

display(spark.createDataFrame(df_baseline_results))

# COMMAND ----------

## 7. Definição dos modelos supervisionados / Supervised model definition

Criamos pipelines com imputação, padronização quando necessário e o estimador final.

We create pipelines with imputation, scaling when needed and the final estimator.

In [0]:
 ## 7. Definição dos modelos supervisionados / Supervised model definition

# COMMAND ----------

models = {}

# Linear models need scaling
# Modelos lineares se beneficiam de padronização
models["linear_regression"] = {
    "type": "linear",
    "features": linear_features,
    "estimator": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
}

models["ridge_regression"] = {
    "type": "linear",
    "features": linear_features,
    "estimator": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=42))
    ])
}

models["lasso_regression"] = {
    "type": "linear",
    "features": linear_features,
    "estimator": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.001, random_state=42, max_iter=10000))
    ])
}

# Tree-based models do not require scaling
# Modelos baseados em árvore não precisam de padronização
models["random_forest"] = {
    "type": "tree",
    "features": tree_features,
    "estimator": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

if XGBOOST_AVAILABLE:
    models["xgboost"] = {
        "type": "tree",
        "features": tree_features,
        "estimator": Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="reg:squarederror",
                random_state=42
            ))
        ])
    }

if LIGHTGBM_AVAILABLE:
    models["lightgbm"] = {
        "type": "tree",
        "features": tree_features,
        "estimator": Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                verbose=-1
            ))
        ])
    }

print("Modelos configurados / Configured models:")
for name in models.keys():
    print(f"- {name}")

# COMMAND ----------

## 8. Backtesting temporal dos modelos supervisionados / Temporal backtesting for supervised models

Nesta etapa simulamos produção:

- Treina no passado.
- Prevê o próximo mês.
- Anda uma janela para frente.
- Repete o processo.

In this step we simulate production:

- Train on the past.
- Predict the next month.
- Move one window forward.
- Repeat.

In [0]:
 ## 8. Backtesting temporal dos modelos supervisionados / Temporal backtesting for supervised models

# COMMAND ----------

def get_target_series(data):
    """
    Retorna o target conforme configuração TARGET_MODE.
    Returns the target based on TARGET_MODE.
    """
    if TARGET_MODE == "log":
        return data[TARGET_LOG_COL].astype(float)
    return data[TARGET_COL].astype(float)

def run_temporal_backtest(data, model_name, model_config, min_training_months=48):
    """
    Executa backtesting temporal para um modelo.
    Runs temporal backtesting for a model.
    """
    data = data.sort_values("dt_mes_referencia").reset_index(drop=True).copy()
    features = model_config["features"]
    estimator_template = model_config["estimator"]

    predictions = []

    for test_idx in range(min_training_months, len(data)):
        train_data = data.iloc[:test_idx].copy()
        test_data = data.iloc[[test_idx]].copy()

        # Remove rows without target
        # Remove linhas sem target
        train_data = train_data[train_data[TARGET_COL].notna()].copy()
        test_data = test_data[test_data[TARGET_COL].notna()].copy()

        if train_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = get_target_series(train_data)

        X_test = test_data[features]
        y_test_raw = test_data[TARGET_COL].astype(float).values

        # Clone estimator by using sklearn clone
        # Clonar estimador usando sklearn clone
        from sklearn.base import clone
        estimator = clone(estimator_template)

        estimator.fit(X_train, y_train)

        y_pred_model_scale = estimator.predict(X_test)
        y_pred_raw = inverse_target(y_pred_model_scale)

        predictions.append({
            "nm_modelo": model_name,
            "ds_tipo_modelo": model_config["type"],
            "nr_ano_mes_referencia": int(test_data["nr_ano_mes_referencia"].iloc[0]),
            "dt_mes_referencia": test_data["dt_mes_referencia"].iloc[0],
            "nr_ano_mes_previsao": int(test_data["nr_ano_mes_previsao"].iloc[0]),
            "dt_mes_previsao": test_data["dt_mes_previsao"].iloc[0],
            "vl_sinistro_real": float(y_test_raw[0]),
            "vl_sinistro_previsto": float(y_pred_raw[0]),
            "vl_erro": float(y_test_raw[0] - y_pred_raw[0]),
            "vl_erro_abs": float(abs(y_test_raw[0] - y_pred_raw[0])),
            "pc_erro": float((y_test_raw[0] - y_pred_raw[0]) / y_test_raw[0]) if y_test_raw[0] != 0 else np.nan,
            "qt_linhas_treino": int(len(train_data)),
            "dh_execucao": datetime.now()
        })

    return pd.DataFrame(predictions)

all_model_predictions = []

for model_name, model_config in models.items():
    print(f"Executando backtest / Running backtest: {model_name}")
    pred_df = run_temporal_backtest(
        data=df_train_all,
        model_name=model_name,
        model_config=model_config,
        min_training_months=MIN_TRAINING_MONTHS
    )
    all_model_predictions.append(pred_df)

df_model_predictions = pd.concat(all_model_predictions, ignore_index=True) if all_model_predictions else pd.DataFrame()

print(f"Predições geradas / Generated predictions: {len(df_model_predictions)}")
display(spark.createDataFrame(df_model_predictions) if not df_model_predictions.empty else None)

# COMMAND ----------

## 9. Consolidação das métricas / Metrics consolidation

Consolidamos os resultados de baselines e modelos supervisionados.

We consolidate baseline and supervised model results.

In [0]:
 ## 9. Consolidação das métricas / Metrics consolidation

# COMMAND ----------

model_results = []

if not df_model_predictions.empty:
    for model_name, group in df_model_predictions.groupby("nm_modelo"):
        metrics = calc_metrics(group["vl_sinistro_real"], group["vl_sinistro_previsto"])

        model_results.append({
            "nm_modelo": model_name,
            "ds_tipo_modelo": group["ds_tipo_modelo"].iloc[0],
            "qt_observacoes_avaliacao": len(group),
            **metrics,
            "dh_execucao": datetime.now()
        })

df_model_results = pd.DataFrame(model_results)

# Standardize baseline prediction columns for union
# Padronizar colunas de predição dos baselines para união
if not df_baseline_predictions.empty:
    df_baseline_predictions_std = df_baseline_predictions.rename(
        columns={TARGET_COL: "vl_sinistro_real"}
    )
    df_baseline_predictions_std = df_baseline_predictions_std[[
        "nm_modelo",
        "ds_tipo_modelo",
        "nr_ano_mes_referencia",
        "dt_mes_referencia",
        "nr_ano_mes_previsao",
        "dt_mes_previsao",
        "vl_sinistro_real",
        "vl_sinistro_previsto",
        "vl_erro",
        "vl_erro_abs",
        "pc_erro",
        "dh_execucao"
    ]]
    df_baseline_predictions_std["qt_linhas_treino"] = np.nan
else:
    df_baseline_predictions_std = pd.DataFrame()

# Union predictions
# União das predições
df_all_predictions = pd.concat(
    [df_baseline_predictions_std, df_model_predictions],
    ignore_index=True
)

# Union results
# União dos resultados
df_all_results = pd.concat(
    [df_baseline_results, df_model_results],
    ignore_index=True
)

# Add ranking
# Adicionar ranking
if not df_all_results.empty:
    df_all_results = df_all_results.sort_values(["vl_wape", "vl_mae"], ascending=True).reset_index(drop=True)
    df_all_results["nr_ranking_wape"] = np.arange(1, len(df_all_results) + 1)

display(spark.createDataFrame(df_all_results))

# COMMAND ----------

## 10. Seleção do modelo campeão / Champion model selection

Selecionamos o melhor modelo pelo menor WAPE.

We select the best model based on the lowest WAPE.

In [0]:
 ## 10. Seleção do modelo campeão / Champion model selection

# COMMAND ----------

if df_all_results.empty:
    raise ValueError("Nenhum resultado de modelo foi gerado.")

champion_row = df_all_results.sort_values(["vl_wape", "vl_mae"], ascending=True).iloc[0]

CHAMPION_MODEL_NAME = champion_row["nm_modelo"]
CHAMPION_MODEL_TYPE = champion_row["ds_tipo_modelo"]

print("Modelo campeão / Champion model")
print(f"Nome / Name: {CHAMPION_MODEL_NAME}")
print(f"Tipo / Type: {CHAMPION_MODEL_TYPE}")
print(f"WAPE: {champion_row['vl_wape']:.4f}")
print(f"MAPE: {champion_row['vl_mape']:.4f}")
print(f"MAE: {champion_row['vl_mae']:.2f}")
print(f"RMSE: {champion_row['vl_rmse']:.2f}")

# COMMAND ----------

## 11. Treino final e previsão do próximo mês / Final training and next month forecast

Treinamos o modelo campeão com toda a base histórica disponível e geramos a previsão final.

We train the champion model using all available history and generate the final forecast.

In [0]:
## 11. Treino final e previsão do próximo mês / Final training and next month forecast

# COMMAND ----------

def predict_baseline_next_month(df_forecast_row, baseline_name):
    """
    Gera previsão usando modelo baseline.
    Generates forecast using baseline model.
    """
    baseline_map = {
        "baseline_ultimo_mes": "vl_pred_baseline_ultimo_mes",
        "baseline_sazonal_ano_anterior": "vl_pred_baseline_sazonal_ano_anterior",
        "baseline_media_3m": "vl_pred_baseline_media_3m",
        "baseline_media_6m": "vl_pred_baseline_media_6m",
        "baseline_media_12m": "vl_pred_baseline_media_12m",
        "baseline_media_ponderada_3m": "vl_pred_baseline_media_ponderada_3m"
    }

    pred_col = baseline_map.get(baseline_name)

    if pred_col is None:
        raise ValueError(f"Baseline não reconhecido / Unknown baseline: {baseline_name}")

    if pred_col not in df_forecast_row.columns:
        raise ValueError(f"Coluna de baseline ausente / Missing baseline column: {pred_col}")

    return float(df_forecast_row[pred_col].iloc[0])

forecast_records = []

if df_forecast.empty:
    print("Sem linha de previsão. Nenhuma previsão final será gerada.")
else:
    forecast_row = df_forecast.iloc[[0]].copy()

    if CHAMPION_MODEL_TYPE == "baseline":
        final_prediction = predict_baseline_next_month(forecast_row, CHAMPION_MODEL_NAME)
        final_model_object = None
        final_feature_list = []
    else:
        model_config = models[CHAMPION_MODEL_NAME]
        final_feature_list = model_config["features"]

        train_data = df_train_all.copy()
        train_data = train_data[train_data[TARGET_COL].notna()].copy()

        X_train_final = train_data[final_feature_list]
        y_train_final = get_target_series(train_data)

        X_forecast = forecast_row[final_feature_list]

        from sklearn.base import clone
        final_model_object = clone(model_config["estimator"])
        final_model_object.fit(X_train_final, y_train_final)

        pred_model_scale = final_model_object.predict(X_forecast)
        final_prediction = float(inverse_target(pred_model_scale)[0])

    forecast_records.append({
        "nr_ano_mes_referencia": int(forecast_row["nr_ano_mes_referencia"].iloc[0]),
        "dt_mes_referencia": forecast_row["dt_mes_referencia"].iloc[0],
        "nr_ano_mes_previsao": int(forecast_row["nr_ano_mes_previsao"].iloc[0]),
        "dt_mes_previsao": forecast_row["dt_mes_previsao"].iloc[0],
        "nm_modelo_campeao": CHAMPION_MODEL_NAME,
        "ds_tipo_modelo": CHAMPION_MODEL_TYPE,
        "vl_sinistro_previsto": final_prediction,
        "vl_wape_backtest": float(champion_row["vl_wape"]),
        "vl_mape_backtest": float(champion_row["vl_mape"]),
        "vl_mae_backtest": float(champion_row["vl_mae"]),
        "vl_rmse_backtest": float(champion_row["vl_rmse"]),
        "qt_observacoes_avaliacao": int(champion_row["qt_observacoes_avaliacao"]),
        "dh_execucao": datetime.now()
    })

df_forecast_final = pd.DataFrame(forecast_records)

display(spark.createDataFrame(df_forecast_final) if not df_forecast_final.empty else None)

# COMMAND ----------

## 12. Registro opcional no MLflow / Optional MLflow logging

Registramos métricas e, quando o campeão for supervisionado, também registramos o modelo.

We log metrics and, when the champion is supervised, also log the model.

In [0]:
# 12. Registro opcional no MLflow / Optional MLflow logging

# COMMAND ----------

if MLFLOW_AVAILABLE:
    mlflow.set_experiment(EXPERIMENT_NAME)

    with mlflow.start_run(run_name=f"susep_sinistro_mensal_{CHAMPION_MODEL_NAME}"):

        mlflow.log_param("source_table", SOURCE_TABLE)
        mlflow.log_param("target_col", TARGET_COL)
        mlflow.log_param("target_mode", TARGET_MODE)
        mlflow.log_param("champion_model", CHAMPION_MODEL_NAME)
        mlflow.log_param("champion_model_type", CHAMPION_MODEL_TYPE)
        mlflow.log_param("min_training_months", MIN_TRAINING_MONTHS)

        mlflow.log_metric("wape", float(champion_row["vl_wape"]))
        mlflow.log_metric("mape", float(champion_row["vl_mape"]))
        mlflow.log_metric("mae", float(champion_row["vl_mae"]))
        mlflow.log_metric("rmse", float(champion_row["vl_rmse"]))

        if not df_forecast_final.empty:
            mlflow.log_metric("forecast_value", float(df_forecast_final["vl_sinistro_previsto"].iloc[0]))

        # Log supervised model only
        # Registrar modelo supervisionado apenas
        if CHAMPION_MODEL_TYPE != "baseline" and final_model_object is not None:
            mlflow.sklearn.log_model(
                sk_model=final_model_object,
                artifact_path="model"
            )

        print("MLflow registrado com sucesso / MLflow logged successfully")
else:
    print("MLflow não disponível. Etapa ignorada / MLflow not available. Step skipped.")

# COMMAND ----------

## 13. Gravação dos resultados em Delta / Write results to Delta

Gravamos:

1. tabela de métricas por modelo;
2. tabela de predições do backtesting;
3. tabela com a previsão final do próximo mês.

We write:

1. model metrics table;
2. backtesting predictions table;
3. final next-month forecast table.

In [0]:
 ## 13. Gravação dos resultados em Delta / Write results to Delta

# COMMAND ----------

# Convert Pandas to Spark
# Converter Pandas para Spark

if not df_all_results.empty:
    spark_results = spark.createDataFrame(df_all_results)
    (
        spark_results
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RESULTS_TABLE)
    )
    print(f"Tabela gravada / Table written: {RESULTS_TABLE}")

if not df_all_predictions.empty:
    spark_predictions = spark.createDataFrame(df_all_predictions)
    (
        spark_predictions
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(PREDICTIONS_TABLE)
    )
    print(f"Tabela gravada / Table written: {PREDICTIONS_TABLE}")

if not df_forecast_final.empty:
    spark_forecast = spark.createDataFrame(df_forecast_final)
    (
        spark_forecast
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FORECAST_TABLE)
    )
    print(f"Tabela gravada / Table written: {FORECAST_TABLE}")

# COMMAND ----------

## 14. Visualização final dos resultados / Final result visualization

Consultas finais para análise executiva.

Final queries for executive analysis.

In [0]:
## 14. Visualização final dos resultados / Final result visualization

# COMMAND ----------

display(
    spark.table(RESULTS_TABLE)
    .orderBy("nr_ranking_wape")
)

display(
    spark.table(FORECAST_TABLE)
)

# COMMAND ----------

## 15. Consulta SQL sugerida para dashboard / Suggested SQL query for dashboard

Use esta consulta para visualizar real vs previsto no backtesting do modelo campeão.

Use this query to visualize actual vs predicted values for the champion model backtesting.

In [0]:
%sql
 SELECT
     p.nr_ano_mes_previsao,
     p.dt_mes_previsao,
     p.nm_modelo,
     p.ds_tipo_modelo,
     p.vl_sinistro_real,
     p.vl_sinistro_previsto,
     p.vl_erro,
     p.vl_erro_abs,
     p.pc_erro
 FROM susep_gold.tb_ml_sinistro_mensal_predictions p
 INNER JOIN susep_gold.tb_ml_sinistro_mensal_model_results r
     ON p.nm_modelo = r.nm_modelo
 WHERE r.nr_ranking_wape = 1
 ORDER BY p.dt_mes_previsao DESC
 limit 12

# COMMAND ----------

## 16. Observações importantes / Important notes

### Português

- O notebook usa backtesting temporal, não split aleatório.
- A última linha da wide table, com `fl_linha_previsao = 1`, é usada para prever o próximo mês.
- O modelo campeão é escolhido pelo menor WAPE.
- O uso de `vl_premio_prox_mes_aux` como feature foi evitado para não gerar vazamento de informação.
- Como a base mensal é pequena, modelos complexos podem sofrer overfitting.
- O baseline sazonal pode ser muito competitivo em série mensal.

### English

- The notebook uses temporal backtesting, not random split.
- The last row from the wide table, marked as `fl_linha_previsao = 1`, is used to forecast the next month.
- The champion model is selected based on the lowest WAPE.
- `vl_premio_prox_mes_aux` is not used as a feature to avoid data leakage.
- Since the monthly dataset is small, complex models may overfit.
- The seasonal baseline can be very competitive for monthly time series.